# Can you trust the instrument's spectrum? Process the raw data yourself

You'll turn the raw signal the instrument records into a spectrum, compare it with the instrument's
own, and decide whether you can trust the instrument's spectrum.

The spectrometer records a **free induction decay** (FID): signal against time. Here you process
the Ala-Ala FIDs from notebook 01 into spectra yourself, one step per cell.

Time: about 30 minutes. Do notebook 01 first. Boxes headed **RECORD YOUR ANSWER** are where you
write something; the reveal then shows what happened.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0. Paper: *Scientific Reports* 14, 17083 (2024).

In [ ]:
# Setup. In Colab this first installs the training package and the Rakiura helper (about 30 seconds).
import importlib.util, os
if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"
import matplotlib.pyplot as plt
import nmrtrain as nt

NOTEBOOK = "notebooks/04_process_raw_fids.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

> **CONNECT TO RAKIURA (optional, recommended)**
>
> **What it does:** ticks your checklist on the step as you go and attaches your figures to it. Nothing is sent until you connect.
>
> **How:** run the cell below, open the link it prints, check the code matches the one printed here, and click Allow. In Colab you do this once per session. To skip the click, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; get the key in Rakiura under Settings > Notebook key).
>
> **Skipping it is fine:** your work is kept on this computer, and you can attach your files on the step page instead.

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = nt.portal.connect(NOTEBOOK, sign_in=True)

In [ ]:
paths = nt.data.fetch("11223995", "AA", acid="K", nucleus="1H", times=["1day", "4Month"])
dic, fid, info = nt.io.load_fid(paths[0])
print(f"{info.label}: {fid.size} complex points, sweep width {dic['acqus']['SW_h']:.0f} Hz")

fig_fid, ax = plt.subplots(figsize=(10, 3))
ax.plot(fid.real[:4000], lw=0.6)
ax.set_xlabel("time (points)"); ax.set_yticks([]); ax.set_title("the raw FID (first 4000 points)")
plt.show()

### What you need

The instrument made its own spectrum from this same raw signal, with the same steps you are about
to run, so a careful run of your own should match it.

These spectra come from a Bruker spectrometer. With its data, the Fourier transform (the step that
turns the signal over time into peaks by frequency) puts high and low ppm at opposite ends, so the
spectrum comes out mirrored. In 98% acid the acid peak sits near 11.5 ppm. Mirroring flips the axis
end to end around its middle, about 6 ppm on this scale.

Longer version: NMR project step 3, Can you trust the instrument's spectrum? Process the raw data yourself.

> **RECORD YOUR ANSWER: predict before you process anything**
>
> **What gets saved:** `MY_PREDICTION`, two short answers, saved with the time.
>
> **Where to look:** the paragraph above, and the acid peak you found in notebook 01.
>
> **How to fill this in:**
> - `matches_instrument`: will your spectrum match the instrument's: `"yes"`, `"no"` or `"unsure"`
> - `acid_ppm_if_not_reversed`: where the acid peak lands, in ppm, if you skip reversing the axis (step 6 below), a number like `3.0`

<details><summary>Stuck? Open hint 1</summary>

How far is 11.5 ppm from the middle of the axis at about 6 ppm? Go the same distance on the other side.

</details>

<details><summary>Stuck? Open hint 2</summary>

11.5 is about 5.5 above 6, so the mirror image lands about 5.5 below 6, between 0.5 and 0.6 ppm.

</details>

In [ ]:
MY_PREDICTION = {
    "matches_instrument": ...,         # "yes", "no" or "unsure"
    "acid_ppm_if_not_reversed": ...,   # a number in ppm: where the acid peak lands if you skip the reverse step
}
nt.record.predict("raw", MY_PREDICTION)

## The steps

The next two cells run steps 1 to 5:

1. Remove the digital filter delay: the instrument's digital filter delays the start of the
   recording, and this step removes the delay.
2. Line broadening: multiply the signal by a decaying curve. That smooths the noise and widens each
   peak slightly. The instrument used 0.5 Hz, which widens each peak by about that much.
3. Zero fill: pad the recording with zeros, so the spectrum has more points and peaks come out
   smoother. It adds no information.
4. Fourier transform: turn the signal over time into peaks by frequency.
5. Phase: make every peak point up, with its two sides symmetric (automatic here).

In [ ]:
f1 = nt.process.remove_digital_filter(dic, fid)
f2 = nt.process.line_broaden(dic, f1, lb_hz=0.5)

### What you need

The recording has 32,768 points. Zero-filling pads it with zeros before the transform. That gives
more points per peak (a finer ppm grid), so peaks come out smoother. It adds no new information. The
instrument's own spectrum has 32,768 points, so it padded nothing.

Longer version: NMR project step 3, Can you trust the instrument's spectrum? Process the raw data yourself.

> **RECORD YOUR ANSWER: choose the zero-fill**
>
> **What gets saved:** `ZERO_FILL`, the number of points after padding.
>
> **Where to look:** the overlay plots and the numbers in the reveal below, after you run it.
>
> **How to fill this in:**
> - Keep `65536` (twice the recording) for the first run.
> - Then try `32768`, the instrument's choice (no padding), and `131072`. Each time, run the cells down to the reveal again and compare the peak shapes and the largest line difference.

<details><summary>Stuck? Open hint 1</summary>

Run once with each value and look at the tops of the peaks in the reveal plots.

</details>

<details><summary>Stuck? Open hint 2</summary>

With fewer points, peaks look more jagged and their tops can fall between points. More points give a
finer grid but no new information. A value below 32768 has nothing to pad, so the notebook keeps the
recording as it is.

</details>

In [ ]:
ZERO_FILL = 65536   # the number of points after padding: 65536, or try 32768 or 131072
nt.record.saved("ZERO_FILL", ZERO_FILL, "the number of points after zero-filling")
f3 = nt.process.zero_fill(f2, ZERO_FILL)
spec = nt.process.fft(f3)
spec = nt.process.autophase(spec)

### Trap: the mirrored axis

The next cell checks your spectrum before the last steps. Compare where the acid peak lands with
your prediction.

In [ ]:
ppm = nt.process.ppm_axis(dic, spec)
not_reversed = info.with_data(ppm, spec.real, processing="raw")
print(nt.checks.check_axis(not_reversed))

The next cell runs the last two steps and checks the axis again:

6. Reverse: with this instrument's data the transform puts high and low ppm at opposite ends; this
   step puts them back.
7. Baseline: subtract a smooth curve so the floor between peaks sits at zero.

In [ ]:
mine = info.with_data(ppm, nt.process.baseline(nt.process.reverse(spec).real), processing="raw")
print(nt.checks.check_axis(mine))

## Reveal: your spectrum over the instrument's

The next cell processes the 1 day and 4 month FIDs your way and lays each over the instrument's
spectrum. For each file it prints:

- where your acid peak sits, and where the instrument's does
- the largest difference in position between matching lines between 1 and 6.5 ppm, and the line it
  belongs to. Each line of a split peak (two close lines for a methyl) counts on its own. Lines are
  compared, not the centres of whole peaks, because a centre worked out from line heights moves with
  your processing choices.
- the similarity: the correlation of the two traces between 1 and 6.5 ppm, where 1 means the same
  shape and 0.95 or more counts as a match

In [ ]:
fig_overlay, sims, lines, acid_peaks = None, {}, {}, {}
if nt.record.revealed("raw"):
    fig_overlay, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    for ax, path in zip(axes, paths):
        inst = nt.load_processed(path)
        own = nt.process.process_fid(path, size=ZERO_FILL)
        sims[inst.time] = nt.peaks.similarity(own, inst)
        lines[inst.time] = nt.peaks.line_differences(own, inst)
        acid_peaks[inst.time] = {"mine": round(nt.checks.acid_ppm(own), 3), "instrument": round(nt.checks.acid_ppm(inst), 3)}
        nt.plot.stack([inst, own], 1.0, 6.5, norm=(1.0, 6.5), offset=0.0, ax=ax,
                      labels=["instrument", "mine"], title=inst.label)
        print(f"{inst.label}: acid peak at {acid_peaks[inst.time]['mine']:.3f} ppm in yours, "
              f"{acid_peaks[inst.time]['instrument']:.3f} ppm in the instrument's")
        print("  ", nt.checks.line_feedback(lines[inst.time]))
        print("  ", nt.checks.similarity_feedback(sims[inst.time]))
    plt.show()

### What you need

You can trust the instrument's spectrum of a file if your lines sit where its lines do, and your acid
peak sits where its acid peak does. Small differences in height, and in the floor between peaks, come
from baseline and phase choices; they do not move any line. Pick one:

- **Yes:** your lines and your acid peak sit where the instrument's do.
- **No:** some lines sit in different places. Say which, and by how much.
- **Unsure:** say what you could not check.

Longer version: NMR project step 3, Can you trust the instrument's spectrum? Process the raw data yourself.

> **RECORD YOUR ANSWER: can you trust the instrument's spectrum of this file?**
>
> **What gets saved:** `TRUST`, your decision, and `TRUST_NOTE`, the evidence for it in a sentence.
>
> **Where to look:** the acid peaks, line differences and similarities the reveal printed above.
>
> **How to fill this in:**
> - `TRUST`: `"yes"`, `"no"` or `"unsure"`, as defined above
> - `TRUST_NOTE`: the numbers your decision rests on, in your own words, or `""`

<details><summary>Stuck? Open hint 1</summary>

Compare the largest line difference with the gap between the two lines of one methyl peak, about
0.016 ppm. Is it much smaller?

</details>

<details><summary>Stuck? Open hint 2</summary>

If every line sits within a few thousandths of a ppm of the instrument's and the two acid peaks
agree, write "yes".

</details>

In [ ]:
TRUST = ""        # your decision: "yes", "no" or "unsure"
TRUST_NOTE = ""   # the numbers your decision rests on, or ""
if TRUST in ("yes", "no", "unsure"):
    nt.record.saved("TRUST", TRUST, "your decision on the instrument's spectrum")
else:
    print(f'Not saved yet TRUST = {TRUST!r}: write "yes", "no" or "unsure".')
nt.record.saved("TRUST_NOTE", TRUST_NOTE, "the evidence for your decision");

> **SAVE YOUR HAND-IN: your processing hand-in**
>
> **What gets saved:** in `handin/`: the overlay and `verdict.json` (your prediction, your decision and its evidence, the acid peaks, the largest line differences, the similarities and your zero-fill).
>
> **Where to look:** nothing new to fill in: the cell uses what you ran above.
>
> **How to fill this in:**
> - Run the cell. These files are your hand-in for NMR project step 3: the overlay is your image, and `verdict.json` holds your note.
> - Nothing is sent for review until you run `nt.portal.submit()` in the last line of the cell.

<details><summary>Stuck? Open hint 1</summary>

If it saves no figure, run the reveal cell above first, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin("raw-overlay", figures={"overlay": fig_overlay} if fig_overlay else {},
                          verdict={"prediction": MY_PREDICTION, "decision": TRUST, "note": TRUST_NOTE,
                                   "acid_peak_ppm": acid_peaks, "line_differences": lines,
                                   "similarity": sims, "zero_fill": ZERO_FILL},
                          numbers={"min_similarity": round(min(sims.values()), 3) if sims else "blank",
                                   "max_line_diff_ppm": max((d["largest_ppm"] for d in lines.values()
                                                             if d["largest_ppm"] is not None), default="blank"),
                                   "trust": TRUST or "blank"},
                          question="raw", processing="raw")
# nt.portal.submit()